In [ ]:
import os
from pathlib import Path

import pandas as pd
import pygwalker as pyg

from dotenv import load_dotenv

from sqlalchemy import create_engine
from sqlalchemy import text

Note: you may need to restart the kernel to use updated packages.


In [ ]:
PROJECT_ROOT = Path.cwd()

ENV_FILE = PROJECT_ROOT / ".env"

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f".env no encontrado: {ENV_FILE}"
    )

load_dotenv(
    ENV_FILE,
    override=True
)

SERVER = os.getenv("DB_SERVER")
DATABASE = os.getenv("DB_NAME")
DRIVER = os.getenv(
    "DB_DRIVER",
    "ODBC Driver 17 for SQL Server"
)

USE_TRUSTED = (
    os.getenv(
        "DB_USE_TRUSTED_CONNECTION",
        "false"
    )
    .lower()
    in ("true", "1", "yes")
)

if not SERVER:
    raise RuntimeError(
        "DB_SERVER no configurado."
    )

if not DATABASE:
    raise RuntimeError(
        "DB_NAME no configurado."
    )

✅ Variables de entorno cargadas desde: D:\01.Datos_AlbertoDzib\Portafolio-Alberto\P4_Real_World_Ingestion\.env
🔐 Intentando conectar con Windows Authentication a DZIBPC/P4_Global_SupplyChain usando driver ODBC Driver 17 for SQL Server...
✅ Conexión a SQL Server correcta.
📊 Extrayendo datos desde SQL Server...
⚠️ La conexión a SQL Server sí funciona, pero las vistas analíticas no están disponibles. Se usará un conjunto de datos de ejemplo para continuar.
Detalle técnico: Execution failed on sql 'SELECT * FROM Analytics.vw_Shipping_Efficiency': (pyodbc.ProgrammingError) ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Analytics.vw_Shipping_Efficiency'. (208) (SQLExecDirectW)")
[SQL: SELECT * FROM Analytics.vw_Shipping_Efficiency]
(Background on this error at: https://sqlalche.me/e/21/f405)
✅ DataFrames listos para la dashboard.
Logística: 10 filas, 6 columnas
Rendimiento: 8 filas, 5 columnas
       fecha  region   ruta  envios  tiempo_promedio

In [ ]:
connection_string = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "TrustServerCertificate=yes;"
)

if USE_TRUSTED:

    connection_string += (
        "Trusted_Connection=yes;"
    )

else:

    raise RuntimeError(
        "Solo se soporta autenticación integrada."
    )

engine = create_engine(
    "mssql+pyodbc://",
    creator=lambda:
        __import__("pyodbc").connect(
            connection_string
        )
)

DB_SERVER=DZIBPC
DB_NAME=P4_Global_SupplyChain
DB_DRIVER=ODBC Driver 17 for SQL Server



In [ ]:
with engine.connect() as conn:

    metadata = conn.execute(
        text(
            """
            SELECT
                @@SERVERNAME AS ServerName,
                DB_NAME() AS DatabaseName
            """
        )
    ).fetchone()

print(
    f"""
Servidor : {metadata.ServerName}

Base     : {metadata.DatabaseName}
"""